In [1]:
# Table 1: the five census-relevant genes (Day 8). Two strict chaperone/protease
# census members found in the regulon (dnj-10, ymel-1) plus the three borderline
# genes the census decision named and checked but excluded on domain grounds
# (prx-19, cbp-3, tspo-1). Binding is reported as three separate columns - published
# (Soo & Van Raamsdonk's own ChIP column), Nargund 2015's independently deposited
# bound-gene list, and this project's own operon-aware peak-to-gene reassignment -
# rather than collapsed into one, since the three do not always agree and collapsing
# them would hide that.
#
# Reads regulon ranks/scores from results/regulon_61.csv (already computed and
# validated in table_s2.ipynb) and re-derives the two things that were not yet
# assembled anywhere: the borderline genes' real Pfam domain text (read directly off
# the annotation file, not from prose) and Nargund 2015 / this-study binding status
# for prx-19, cbp-3, and tspo-1, which binding.ipynb and analysis_c.ipynb checked for
# dnj-10 and ymel-1 but had no reason to check for the other three at the time.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd

TARGET_GENES = ["dnj-10", "ymel-1", "prx-19", "cbp-3", "tspo-1"]

regulon = pd.read_csv("results/regulon_61.csv")
if len(regulon) != 61:
    raise RuntimeError(f"results/regulon_61.csv has {len(regulon)} rows, expected 61.")

rows = regulon[regulon["public_name"].isin(TARGET_GENES)].set_index("public_name")
if sorted(rows.index) != sorted(TARGET_GENES):
    raise RuntimeError(f"Expected all 5 target genes in the regulon, found {sorted(rows.index)}.")

print(rows[["seqname", "score", "score_var", "rank_score", "rank_var", "soo_bound_published"]].to_string())

              seqname       score  score_var  rank_score  rank_var soo_bound_published
public_name                                                                           
tspo-1        C41G7.9  382.422885  18.225206          42        21                 Yes
cbp-3        F40F12.7  325.327236   8.422587          44        57                  No
dnj-10        F22B7.5  306.680798  17.141011          45        23                  No
prx-19        F54F2.8  239.452749  18.344035          54        20                 Yes
ymel-1       M03C11.5  118.842331   8.101193          61        58                  No


In [2]:
# Pfam domain text for the two strict census members, from the frozen census file
# (already validated to reproduce exactly in census_build.ipynb).
census = pd.read_csv("data/chaperone_protease_census.csv")
census_domains = census.set_index("public_name")["pfam_families"].to_dict()

# The three borderline genes carry no domain from the census inclusion list at all
# (gate_decisions.md, "Permissive-rule addendum") - that is why they are excluded.
# Their actual domains are read directly off the same protein-annotation file
# census_build.ipynb uses, rather than typed from memory, so the table cannot carry
# a repeat of the FtsH_AAA/Peptidase_M41-style naming error found there.
PROTEIN_GFF = "data/raw/c_elegans.PRJNA13758.WS285.protein_annotation.gff3.gz"
BORDERLINE_PROTEINS = {
    "prx-19": ["F54F2.8"],
    "cbp-3": ["F40F12.7"],
    "tspo-1": ["C41G7.9a", "C41G7.9b"],
}
protein_to_gene = {p: g for g, ps in BORDERLINE_PROTEINS.items() for p in ps}

borderline_domains = {}
with gzip.open(PROTEIN_GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[1] != "Pfam" or f[0] not in protein_to_gene:
            continue
        # "Name=Pfam PF04614 (Pex19)" - the parenthesised family name is what the
        # census file stores for the strict members, so borderline entries use the
        # same format for a like-for-like table column.
        name_field = f[8].split("Name=Pfam ")[1].split(";")[0]
        gene = protein_to_gene[f[0]]
        borderline_domains.setdefault(gene, set()).add(name_field)

EXPECTED_BORDERLINE_DOMAINS = {
    "prx-19": {"PF04614 (Pex19)"},
    "cbp-3": {"PF02135 (zf-TAZ)"},
    "tspo-1": {"PF03073 (TspO_MBR)"},
}
for gene, expected in EXPECTED_BORDERLINE_DOMAINS.items():
    got = borderline_domains.get(gene, set())
    if got != expected:
        raise RuntimeError(f"{gene}: expected domain {expected}, got {got} - check the annotation file.")

domain_text = {**census_domains, **{g: ", ".join(sorted(d)) for g, d in borderline_domains.items()}}
for gene in TARGET_GENES:
    print(f"{gene:8s}: {domain_text[gene]}")

dnj-10  : DnaJ
ymel-1  : Peptidase_M41
prx-19  : PF04614 (Pex19)
cbp-3   : PF02135 (zf-TAZ)
tspo-1  : PF03073 (TspO_MBR)


In [3]:
# Nargund 2015 bound-gene lookup, all 5 genes. Same search method as
# binding.ipynb cell 0 (public name, sequence name, and any historical alias,
# searched across every column since the table predates current WormBase
# nomenclature) - binding.ipynb only ran it for the 4 chaperone/QC genes named in
# the roadmap at the time; the three borderline genes were not part of that check
# and are added here rather than assumed.
nargund_df = pd.read_excel("data/raw/nargund2015_TableS1-S2.xlsx")
if nargund_df.shape[0] != 511:
    raise RuntimeError(f"Nargund 2015 table has {nargund_df.shape[0]} rows, expected 511 - file changed?")

ALIASES = {
    "dnj-10": ["dnj-10", "F22B7.5"],
    "ymel-1": ["ymel-1", "yme-1", "M03C11.5"],
    "prx-19": ["prx-19", "F54F2.8"],
    "cbp-3": ["cbp-3", "F40F12.7"],
    "tspo-1": ["tspo-1", "C41G7.9"],
}

nargund_bound = {}
for gene, terms in ALIASES.items():
    found = False
    for term in terms:
        matches = nargund_df.apply(
            lambda col: col.astype(str).str.contains(f"^{term}$|\\b{term}\\b", case=False, regex=True, na=False)
        ).any(axis=1).sum()
        if matches > 0:
            nargund_bound[gene] = True
            found = True
            break
    if not found:
        nargund_bound[gene] = False

# Cross-check against the two already validated in gate_decisions.md before trusting
# the three new lookups run by the same code path.
EXPECTED_NARGUND = {"dnj-10": False, "ymel-1": True}
for gene, expected in EXPECTED_NARGUND.items():
    if nargund_bound[gene] != expected:
        raise RuntimeError(f"{gene}: Nargund 2015 lookup gave {nargund_bound[gene]}, expected {expected}.")

for gene in TARGET_GENES:
    print(f"{gene:8s}: Nargund 2015 bound = {nargund_bound[gene]}")

dnj-10  : Nargund 2015 bound = False
ymel-1  : Nargund 2015 bound = True
prx-19  : Nargund 2015 bound = True
cbp-3   : Nargund 2015 bound = False
tspo-1  : Nargund 2015 bound = True


/opt/homebrew/Caskroom/miniforge/base/envs/atfs1/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [4]:
# This-study binding call, all 5 genes - the identical operon-aware peak-to-gene
# assignment as analysis_c.ipynb (2kb window, own-or-operon-head TSS), re-derived
# here rather than read from a stored intermediate so this notebook stays
# self-validating against the same known-true values before anything downstream
# trusts it. analysis_c.ipynb itself is untouched (frozen).
GFF = "data/raw/c_elegans.PRJNA13758.WS285.annotations.gff3.gz"

genes = {}
with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[1] != "WormBase" or f[2] != "gene":
            continue
        attrs = dict(kv.split("=", 1) for kv in f[8].split(";") if "=" in kv)
        gid_raw = attrs.get("ID", "")
        gid = gid_raw.replace("Gene:", "") if gid_raw.startswith("Gene:") else gid_raw
        if not gid.startswith("WBGene"):
            continue
        genes[gid] = {
            "chr": f[0], "start": int(f[3]), "end": int(f[4]), "strand": f[6],
            "seqname": attrs.get("sequence_name", ""),
            "public_name": attrs.get("locus", ""),
        }
for g in genes.values():
    g["tss"] = g["start"] if g["strand"] == "+" else g["end"]

operons = {}
with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[1] != "operon" or f[2] != "operon":
            continue
        attrs = dict(kv.split("=", 1) for kv in f[8].split(";") if "=" in kv)
        name = attrs.get("Name")
        members = attrs.get("genes", "").split(",") if attrs.get("genes") else []
        if name and members:
            operons[name] = members

gene_to_head = {m: members[0] for members in operons.values() for m in members[1:]}
for gid, g in genes.items():
    g["operon_role"] = "downstream" if gid in gene_to_head else "head_or_independent"
    if g["operon_role"] == "downstream" and gene_to_head[gid] in genes:
        g["head_tss"] = genes[gene_to_head[gid]]["tss"]
        g["head_chr"] = genes[gene_to_head[gid]]["chr"]
    else:
        g["head_tss"] = None

peaks = pd.read_csv("data/liftover/peaks_ce11.bed", sep="\t", header=None,
                     names=["chr", "start", "end", "name"])
peaks["chr"] = peaks["chr"].str.replace("^chr", "", regex=True)

by_chr_own, by_chr_head = {}, {}
for gid, g in genes.items():
    by_chr_own.setdefault(g["chr"], []).append((gid, g["tss"]))
    if g["operon_role"] == "downstream" and g["head_tss"] is not None:
        by_chr_head.setdefault(g["head_chr"], []).append((gid, g["head_tss"]))

WINDOW = 2000
bound = set()
for _, p in peaks.iterrows():
    for gid, tss in by_chr_own.get(p["chr"], []):
        if (p["start"] - WINDOW) <= tss <= (p["end"] + WINDOW):
            bound.add(gid)
    for gid, tss in by_chr_head.get(p["chr"], []):
        if (p["start"] - WINDOW) <= tss <= (p["end"] + WINDOW):
            bound.add(gid)

name_to_gid = {g["public_name"].lower(): gid for gid, g in genes.items() if g.get("public_name")}
EXPECTED_BOUND = {"hsp-6": True, "hsp-60": True, "dnj-10": False, "ymel-1": True}
ok = True
for name, exp in EXPECTED_BOUND.items():
    got = name_to_gid.get(name) in bound
    if got != exp:
        ok = False
    print(f"  validate {name}: expected {exp}, got {got} [{'OK' if got==exp else 'MISMATCH'}]")
if not ok:
    raise RuntimeError("Peak assignment failed validation - do not trust downstream results.")

this_study_bound = {gene: (name_to_gid.get(gene) in bound) for gene in TARGET_GENES}
print()
for gene in TARGET_GENES:
    print(f"{gene:8s}: this-study bound = {this_study_bound[gene]}")

# The one documented disagreement among these 5 (gate_decisions.md, "Binding
# reconciliation") is ymel-1: Soo's column says no, this pipeline and Nargund 2015
# both say yes. Confirm nothing else disagrees silently.
soo_bound = {g: (rows.loc[g, "soo_bound_published"] == "Yes") for g in TARGET_GENES}
disagreements = [g for g in TARGET_GENES if soo_bound[g] != this_study_bound[g]]
if disagreements != ["ymel-1"]:
    raise RuntimeError(f"Expected only ymel-1 to disagree with Soo's column, got {disagreements}.")
print("\nConfirmed: ymel-1 is the only one of the 5 where this-study binding disagrees with Soo's column.")

  validate hsp-6: expected True, got True [OK]
  validate hsp-60: expected True, got True [OK]
  validate dnj-10: expected False, got False [OK]
  validate ymel-1: expected True, got True [OK]

dnj-10  : this-study bound = False
ymel-1  : this-study bound = True
prx-19  : this-study bound = True
cbp-3   : this-study bound = False
tspo-1  : this-study bound = True

Confirmed: ymel-1 is the only one of the 5 where this-study binding disagrees with Soo's column.


In [5]:
# Assemble and write the table. Column order: identity, domain
# evidence and census rule, both ranking metrics (conservative first), then the
# three binding columns kept separate rather than collapsed into one.
CENSUS_RULE = {
    "dnj-10": "Strict census (chaperone)",
    "ymel-1": "Strict census (protease)",
    "prx-19": "Permissive only - excluded, no census domain",
    "cbp-3": "Permissive only - excluded, no census domain",
    "tspo-1": "Permissive only - excluded, no census domain",
}

table1 = pd.DataFrame([
    {
        "public_name": gene,
        "seqname": rows.loc[gene, "seqname"],
        "pfam_domain": domain_text[gene],
        "census_rule": CENSUS_RULE[gene],
        "rank_score_var": int(rows.loc[gene, "rank_var"]),
        "rank_score": int(rows.loc[gene, "rank_score"]),
        "bound_published_soo2021": "Yes" if soo_bound[gene] else "No",
        "bound_nargund2015": "Yes" if nargund_bound[gene] else "No",
        "bound_this_study": "Yes" if this_study_bound[gene] else "No",
    }
    for gene in TARGET_GENES
])

# Final check against the numbers already recorded in gate_decisions.md before
# writing anything out.
EXPECTED_RANKS = {
    "dnj-10": (23, 45), "ymel-1": (58, 61),
    "prx-19": (20, 54), "cbp-3": (57, 44), "tspo-1": (21, 42),
}
for _, r in table1.iterrows():
    exp_var, exp_score = EXPECTED_RANKS[r["public_name"]]
    if (r["rank_score_var"], r["rank_score"]) != (exp_var, exp_score):
        raise RuntimeError(f"{r['public_name']}: rank mismatch against gate_decisions.md.")

os.makedirs("tables", exist_ok=True)
table1.to_csv("results/table_1.csv", index=False)
table1.to_csv("tables/table_1.csv", index=False)

print(table1.to_string(index=False))
print("\nWrote results/table_1.csv and tables/table_1.csv")

public_name  seqname        pfam_domain                                  census_rule  rank_score_var  rank_score bound_published_soo2021 bound_nargund2015 bound_this_study
     dnj-10  F22B7.5               DnaJ                    Strict census (chaperone)              23          45                      No                No               No
     ymel-1 M03C11.5      Peptidase_M41                     Strict census (protease)              58          61                      No               Yes              Yes
     prx-19  F54F2.8    PF04614 (Pex19) Permissive only - excluded, no census domain              20          54                     Yes               Yes              Yes
      cbp-3 F40F12.7   PF02135 (zf-TAZ) Permissive only - excluded, no census domain              57          44                      No                No               No
     tspo-1  C41G7.9 PF03073 (TspO_MBR) Permissive only - excluded, no census domain              21          42                     Yes    